# VisDSR: one overnight diagnostic run

**8 fresh forests × 6 controls × 2 pinned checkpoints = 96 requests.**
Qwen runs first (48), then InternVL (48), in separate processes. Each saved answer refreshes a checksummed backup. The final output includes raw answers, scores, a readable report, and the measured accuracy gate.

Enable **Internet** and **GPU T4 ×2** (a single T4 also supports these checkpoints). Leave the configuration below as supplied. No completed-study ZIP is needed.

**Start with Save Version → Save & Run All.** Wait until the saved version is running; you can then close the browser or sleep your laptop. This runs a fresh copy from top to bottom and retains its output files in the resulting version. Ordinary interactive Run All can lose its files when its runtime is discarded. See [Kaggle's notebook documentation](https://www.kaggle.com/docs/notebooks).

Collection has a seven-hour time budget, leaving time for setup and saving outputs. Runtime errors or the deadline produce an explicitly partial report and backup; correctness is never guaranteed. Attach a downloaded diagnostic backup as a private input dataset to resume this same fixed panel.


In [ ]:
# Configuration: no manual model switch or batch reruns are needed.
COLLECTION_TIME_LIMIT_HOURS = 7  # Greater than 0, at most 8; applies to both models together
BACKUP_INPUT = "auto"  # Or the full path to a diagnostic ZIP/extracted dataset folder
if type(COLLECTION_TIME_LIMIT_HOURS) not in (int, float) or not 0 < COLLECTION_TIME_LIMIT_HOURS <= 8:
    raise ValueError("Choose a collection time limit greater than 0 and at most 8 hours.")
print(f"Both models; 96 planned responses; {COLLECTION_TIME_LIMIT_HOURS}-hour collection budget.")


## Set up and restore

This clones the published implementation, installs pinned inference dependencies, and verifies frozen inputs and independent simulator truths. It checks the GPU before inference. A fresh diagnostic pilot needs no uploaded ZIP.

If continuing a partial diagnostic run, attach its latest backup **before** starting Save & Run All; the separate saved session cannot use files that exist only in your interactive runtime.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

if not Path("/kaggle/working").is_dir():
    raise RuntimeError("Import this notebook into Kaggle and enable a T4 GPU and Internet.")
if "COLLECTION_TIME_LIMIT_HOURS" not in globals() or "BACKUP_INPUT" not in globals():
    raise RuntimeError("Run the Configuration cell first.")
PROJECT = Path("/kaggle/working/VisDSR_diagnostics")
EXPORT = Path("/kaggle/working/visdsr_diagnostics_backup.zip")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/Abhishekgit01/VisDSR.git", str(PROJECT)], check=True)
if not (PROJECT / "diagnostics/pilot/FREEZE.json").is_file():
    raise RuntimeError("This checkout has no diagnostic pilot. Import the latest notebook after the implementation is published.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed",
                "-r", str(PROJECT / "environment/requirements.txt"),
                "-r", str(PROJECT / "environment/requirements-inference.txt")], check=True)
subprocess.run([sys.executable, "-c", "import torch; "
                "assert torch.cuda.is_available(), 'Enable a T4 GPU in Kaggle settings'; "
                "print('CUDA GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"],
               cwd=PROJECT, check=True)
subprocess.run(["make", "build"], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "-m", "diagnostics", "check"], cwd=PROJECT, check=True)

if BACKUP_INPUT == "auto":
    candidates = sorted(Path("/kaggle/input").rglob("*diagnostics*.zip"))
    for manifest in sorted(Path("/kaggle/input").rglob("MANIFEST.json")):
        info = json.loads(manifest.read_text())
        if info.get("protocol_id") == "visdsr-dsu-diagnostics-20261007":
            candidates.append(manifest.parent)
else:
    candidates = [Path(BACKUP_INPUT)]
for backup in candidates:
    subprocess.run([sys.executable, "-m", "diagnostics", "restore", "--input", str(backup)],
                   cwd=PROJECT, check=True)
if not candidates:
    print("No diagnostic backup attached. Starting from any caches already in this session.")
subprocess.run([sys.executable, "-m", "diagnostics", "score"], cwd=PROJECT, check=True)
print("Setup complete. Existing completed-study responses are not included in this pilot.")


## Inspect the fixed inputs

The eight initial maps, operations, and images were checked against the independent simulator reference. The recorded visual inspection covers all four difficulty groups. This preview lets you inspect those same frozen images; it does not generate new inputs.


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

project = Path("/kaggle/working/VisDSR_diagnostics")
pilot = project / "diagnostics/pilot"
if not (pilot / "FREEZE.json").exists():
    raise RuntimeError("Run Set up and restore first.")
tasks = [json.loads(line) for line in (pilot / "tasks.jsonl").read_text().splitlines()]
mapping = json.loads((pilot / "images.json").read_text())
for task in tasks:
    print(task["id"], f"{task['n']} elements / {len(task['operations'])} operations")
    print("Initial:", task["initial"])
    print("Operations:", task["operations"])
    display(Image(filename=str(pilot / mapping[task["id"]]["diagram_image"]), width=512))


## Collect both models automatically

One invocation fills Qwen's 48 requests, releases its process and GPU memory, then fills InternVL's 48 requests. Existing compatible caches are skipped. Invalid and incorrect answers are preserved without content retries. An inference-process failure is recorded, and the other checkpoint can still run within the remaining time budget.

A complete panel prints **COLLECTION COMPLETE: 96/96** and creates **visdsr_diagnostics_complete.zip**, regardless of whether its accuracy gate passes or fails. An incomplete panel prints **PARTIAL COLLECTION** and preserves a resumable **visdsr_diagnostics_backup.zip**. Check the final report rather than assuming that a finished notebook means 96 answers.


In [ ]:
import subprocess
import sys
from pathlib import Path

project = Path("/kaggle/working/VisDSR_diagnostics")
if "COLLECTION_TIME_LIMIT_HOURS" not in globals():
    raise RuntimeError("Run Configuration, then Set up and restore before collecting responses.")
if not (project / "diagnostics/pilot/FREEZE.json").exists():
    raise RuntimeError("Run Set up and restore first.")
subprocess.run([sys.executable, "-u", "-m", "diagnostics", "overnight",
                "--output-dir", "/kaggle/working",
                "--time-limit-hours", str(COLLECTION_TIME_LIMIT_HOURS)],
               cwd=project, check=True)


## Final export and completion check

This final cell refreshes the backup and displays the actual number of saved responses and the measured gate. It runs automatically in Save & Run All.

Tomorrow, open the **completed saved version → Output**, download **visdsr_diagnostics_complete.zip** if it shows **96/96**, and send it for review. If the report is partial, download **visdsr_diagnostics_backup.zip** and attach it to the same notebook for a resumed saved run. Your interactive editor's Output may be empty because the saved version ran in a different session.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

project = Path("/kaggle/working/VisDSR_diagnostics")
export = Path("/kaggle/working/visdsr_diagnostics_backup.zip")
if not (project / "diagnostics/pilot/FREEZE.json").exists():
    raise RuntimeError("Run Set up and restore first.")
subprocess.run([sys.executable, "-m", "diagnostics", "export", "--output", str(export)],
               cwd=project, check=True)
summary_path = project / "results/followup/visdsr-dsu-diagnostics-20261007/summary.json"
summary = json.loads(summary_path.read_text())
print(f"FINAL STATUS: {summary['scored']}/96; accuracy gate={summary['gate']}")
print(f"BACKUP READY: {export}; {export.stat().st_size:,} bytes")
print("Download from the saved version's Output after this run finishes.")
